# Test And Compare Two Teacher Models

Run this notebook after your test arrays exist, for example after the data-preparation cells from the teacher notebook. It loads two saved models, predicts with both, prints metrics, saves confusion matrices, and saves a prediction-by-prediction comparison.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## Paths

In [ ]:
from pathlib import Path
import os

ORIGINAL_MODEL_PATH = Path("/content/drive/MyDrive/VR Emotions (K-Arts, Semi)/vr-face-au-study-crossdomain/outputs/models/teacher-no-eye-AU-natural-acted-tcn.keras")
MODEL_10_PARTICIPANTS_PATH = Path("/content/drive/MyDrive/outputs_10participants/models/teacher-no-eye-AU-natural-acted-tcn.keras")

output_path = Path("/content/drive/MyDrive/outputs_10participants")
predictions_path = output_path / "predictions"
comparison_path = predictions_path / "comparison between 10 part and original"
comparison_path.mkdir(parents=True, exist_ok=True)

print("Original model exists:", ORIGINAL_MODEL_PATH.exists(), ORIGINAL_MODEL_PATH)
print("10-participant model exists:", MODEL_10_PARTICIPANTS_PATH.exists(), MODEL_10_PARTICIPANTS_PATH)
print("Saving results to:", comparison_path)


## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

try:
    from tcn import TCN
except Exception:
    TCN = None

class_names = ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
class_labels = list(range(len(class_names)))


## Choose Test Sets

These variables must already exist from your data-preparation cells. If you only want one test set, remove the others.

In [ ]:
TEST_SETS = {
    "old_natural": (X_test_nat_np, Y_test_nat_np),
    "old_acted_plus_EmojiHero": (X_test_act_comb_np, Y_test_act_comb_np),
    "all_combined": (X_test_comb_np, Y_test_comb_np),
}

# Add the new 10-participant test set if it exists as either numpy arrays or lists.
if "X_test_new_np" in globals() and "Y_test_new_np" in globals():
    TEST_SETS["new_10participants"] = (X_test_new_np, Y_test_new_np)
elif "X_test_new" in globals() and "Y_test_new" in globals():
    X_test_new_np = stack_sequences(X_test_new)
    Y_test_new_np = to_numpy_labels(Y_test_new)
    TEST_SETS["new_10participants"] = (X_test_new_np, Y_test_new_np)

for name, (X, y) in TEST_SETS.items():
    print(name, X.shape, y.shape)


## Load Models

In [ ]:
def load_teacher_model(path):
    custom_objects = {}
    if TCN is not None:
        custom_objects["TCN"] = TCN
    try:
        return tf.keras.models.load_model(path, custom_objects=custom_objects, compile=False, safe_mode=False)
    except TypeError:
        return tf.keras.models.load_model(path, custom_objects=custom_objects, compile=False)

original_model = load_teacher_model(ORIGINAL_MODEL_PATH)
plus10_model = load_teacher_model(MODEL_10_PARTICIPANTS_PATH)

models = {
    "original_teacher": original_model,
    "teacher_plus_10participants": plus10_model,
}

print("Models loaded.")


## Predict, Test, And Compare

In [ ]:
def predict_probs(model, X):
    pred = model.predict(X, verbose=0)
    if isinstance(pred, dict):
        pred = pred.get("y", next(iter(pred.values())))
    elif isinstance(pred, (list, tuple)):
        pred = pred[0]
    return np.asarray(pred)

results = []
all_predictions = []

for test_name, (X_test, y_true) in TEST_SETS.items():
    y_true = np.asarray(y_true).astype(int).reshape(-1)

    test_predictions = {
        "test_set": test_name,
        "sample_index": np.arange(len(y_true)),
        "y_true": y_true,
        "y_true_name": [class_names[i] for i in y_true],
    }

    for model_name, model in models.items():
        probs = predict_probs(model, X_test)
        y_pred = probs.argmax(axis=1)

        results.append({
            "model": model_name,
            "test_set": test_name,
            "n_samples": len(y_true),
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        })

        test_predictions[f"{model_name}_pred"] = y_pred
        test_predictions[f"{model_name}_pred_name"] = [class_names[i] for i in y_pred]
        test_predictions[f"{model_name}_confidence"] = probs.max(axis=1)
        test_predictions[f"{model_name}_correct"] = y_pred == y_true

        print("\n" + "=" * 80)
        print(model_name, "on", test_name)
        print("=" * 80)
        print(classification_report(y_true, y_pred, labels=class_labels, target_names=class_names, zero_division=0))

        cm = confusion_matrix(y_true, y_pred, labels=class_labels)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
        plt.title(f"{model_name} - {test_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(comparison_path / f"confusion_{model_name}_{test_name}.png", dpi=200)
        plt.show()

    pred_df = pd.DataFrame(test_predictions)
    pred_df["same_prediction"] = pred_df["original_teacher_pred"] == pred_df["teacher_plus_10participants_pred"]
    pred_df["original_right_plus10_wrong"] = pred_df["original_teacher_correct"] & ~pred_df["teacher_plus_10participants_correct"]
    pred_df["plus10_right_original_wrong"] = ~pred_df["original_teacher_correct"] & pred_df["teacher_plus_10participants_correct"]
    all_predictions.append(pred_df)

results_df = pd.DataFrame(results)
predictions_df = pd.concat(all_predictions, ignore_index=True)

results_df.to_csv(comparison_path / "model_test_results.csv", index=False)
predictions_df.to_csv(comparison_path / "prediction_comparison.csv", index=False)

display(results_df)
display(predictions_df.head(50))

print("Saved:", comparison_path / "model_test_results.csv")
print("Saved:", comparison_path / "prediction_comparison.csv")


## Simple Score Table

In [ ]:
score_table = results_df.pivot(
    index="test_set",
    columns="model",
    values=["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
)

display(score_table)
score_table.to_csv(comparison_path / "model_test_score_table.csv")
print("Saved:", comparison_path / "model_test_score_table.csv")
